# Golden Power Network Analysis

**Case Study: EU Regulation 2019/452 ("Golden Power")**

A regulation that, despite being classified as "European Union", actually touches on 15 different legislative domains and cites/is cited by 39 laws spread across seemingly unrelated sectors.

This notebook demonstrates:
1. **Analysis of the problem**: How EuroVoc's a priori classification obscures functional relationships
2. **Visualization solution**: How citation-based clustering reveals true legislative architecture

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from itertools import combinations
from collections import Counter
from networkx.algorithms import community

import warnings
warnings.filterwarnings('ignore')

---
## Part 1: Analysis of the Problem

In this section, we analyze how traditional classification systems (EuroVoc) fragment functionally related legislation across multiple administrative domains.

### 1.1 Data Loading and Network Extraction

In [ ]:
# Load data
proc_path = os.path.join('..', 'data', 'processed')
nodes_cleaned = pd.read_csv(os.path.join(proc_path, 'nodes_cleaned.csv'))
edges_cleaned = pd.read_csv(os.path.join(proc_path, 'edges_cleaned.csv'))
concepts_df = pd.read_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'))

# Create concept dictionary
concept_dict = dict(zip(concepts_df['id'].astype(str), concepts_df['label']))

# Define seed regulation (Golden Power)
seed_celex = '32019R0452'
seed_id = seed_celex

# Extract Golden Power ego-network
connections = edges_cleaned[(edges_cleaned['source'] == seed_id) | (edges_cleaned['target'] == seed_id)]
neighbor_ids = set(connections['source']) | set(connections['target'])
golden_power_network = nodes_cleaned[nodes_cleaned['celex'].isin(neighbor_ids)]

print(f"Golden Power Network:")
print(f"  • Connected laws: {len(golden_power_network)}")
print(f"  • Total connections: {len(connections)}")

### 1.2 Distribution of the Network (Bar Chart)

**Problem**: The Golden Power framework is fragmented across 15+ EuroVoc domains, obscuring its thematic unity.

In [ ]:
# Parse and count domains
def parse_domains(domains_str):
    """Parse semicolon-separated domains and clean them"""
    if pd.isna(domains_str) or str(domains_str).strip() == '':
        return ['UNKNOWN']
    domains = [d.strip() for d in str(domains_str).split(';') if d.strip()]
    cleaned = [re.sub(r'^\d+\s+', '', d).strip().upper() for d in domains]
    return cleaned if cleaned else ['UNKNOWN']

# Count domain frequencies
domain_counts = Counter()
for domains_str in golden_power_network['domains'].dropna():
    domain_counts.update(parse_domains(domains_str))

# Get top 15 domains
top_domains = domain_counts.most_common(15)
domains = [d[0] for d in top_domains]
counts = [d[1] for d in top_domains]

# Create bar chart
fig = go.Figure(data=[
    go.Bar(
        x=counts,
        y=domains,
        orientation='h',
        marker=dict(
            color=counts,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Laws')
        )
    )
])

fig.update_layout(
    title='<b>Domain Distribution in Golden Power Network</b><br><sub>Top 15 EuroVoc Domains</sub>',
    xaxis_title='Number of Laws',
    yaxis_title='Domain',
    height=600,
    width=900,
    template='plotly_white',
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

print("\n📊 Key Insight:")
print("Despite being an 'EU' regulation, Golden Power touches 15+ domains.")
print(f"Top 3 domains: {domains[-1]} ({counts[-1]}), {domains[-2]} ({counts[-2]}), {domains[-3]} ({counts[-3]})")

### 1.3 Confusion Matrix: Domain Overlaps

**Problem**: EuroVoc domains are not "silos" - over 60% of laws have multiple domain assignments, creating ambiguity.

In [ ]:
# Prepare data (exclude UNKNOWN)
dom_series = golden_power_network[golden_power_network['domains'] != '00 UNKNOWN']['domains'].dropna()

def get_domain_list(text):
    if ';' in text:
        return [p.strip() for p in text.split(';')]
    elif ',' in text:
        return [p.strip() for p in text.split(',')]
    else:
        return [text.strip()]

# Calculate co-occurrences
pair_counts = Counter()
all_domains = set()

for entry in dom_series:
    domains = get_domain_list(str(entry))
    all_domains.update(domains)
    for pair in combinations(sorted(domains), 2):
        pair_counts[pair] += 1

# Select top domains for matrix
flat_domains = [d for entry in dom_series for d in get_domain_list(str(entry))]
top_domains_list = pd.Series(flat_domains).value_counts().head(15).index.tolist()
unique_domains = sorted(top_domains_list)

# Build co-occurrence matrix
matrix = pd.DataFrame(0, index=unique_domains, columns=unique_domains)

for (d1, d2), count in pair_counts.items():
    if d1 in unique_domains and d2 in unique_domains:
        matrix.loc[d1, d2] = count
        matrix.loc[d2, d1] = count

# Visualize
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(matrix, dtype=bool))

sns.heatmap(matrix, mask=mask, annot=True, fmt='d', cmap='YlGnBu', 
            cbar_kws={'label': 'Shared Laws'})

plt.title('Domain Overlap Matrix: EuroVoc Domain Co-occurrences\n(Golden Power Network)', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/co_occurrence_matrix_final.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Key Insight:")
print("Dense co-occurrence matrix proves domains are highly overlapping.")
print("Example: EU ↔ Law (9 laws), EU ↔ Intl Relations (11), Finance ↔ Trade (8)")

### 1.4 Build Global Graph and Detect Communities

In [ ]:
# Build global graph
print("Building global graph...")
G_global = nx.from_pandas_edgelist(edges_cleaned, source='source', target='target')
print(f"  • Nodes: {G_global.number_of_nodes():,}")
print(f"  • Edges: {G_global.number_of_edges():,}")

# Detect communities using Louvain algorithm
print("\nDetecting functional communities (this may take a moment)...")
clusters_global_list = community.louvain_communities(G_global.to_undirected(), seed=42)

# Create partition dictionary
global_partition = {}
for cluster_id, nodes in enumerate(clusters_global_list):
    for node in nodes:
        global_partition[node] = cluster_id

print(f"  • Communities detected: {len(clusters_global_list)}")

# Extract Golden Power ego-network
print("\nExtracting Golden Power ego-network...")
golden_power_nodes = list(G_global.neighbors(seed_celex)) + [seed_celex]
nodes_golden_power = nodes_cleaned[nodes_cleaned['celex'].isin(golden_power_nodes)].copy()
nodes_golden_power['global_cluster'] = nodes_golden_power['celex'].map(global_partition)

print(f"  • Nodes: {len(nodes_golden_power)}")
print(f"  • Unique communities: {nodes_golden_power['global_cluster'].nunique()}")

# Create subgraph
G_ego = G_global.subgraph(golden_power_nodes).copy()
print(f"  • Edges: {G_ego.number_of_edges()}")

### 1.5 Data Enrichment and Preparation

In [ ]:
# Fill missing data
nodes_golden_power['domains'] = nodes_golden_power['domains'].fillna('').astype(str)
nodes_golden_power['subdomains'] = nodes_golden_power['subdomains'].fillna('').astype(str)

# Manual data correction for missing entries
missing_data = {
    '21994A1223(01)': {'domains': '20 TRADE', 'subdomains': 'Organizzazione mondiale del commercio'},
    '12016E207':      {'domains': '20 TRADE', 'subdomains': 'Politica commerciale comune'},
    '21994A1223(16)': {'domains': '20 TRADE', 'subdomains': 'Accordo generale sugli scambi di servizi'}
}

for celex, info in missing_data.items():
    nodes_golden_power.loc[nodes_golden_power['celex'] == celex, 'domains'] = info['domains']
    nodes_golden_power.loc[nodes_golden_power['celex'] == celex, 'subdomains'] = info['subdomains']

# Parse domains into lists
nodes_golden_power['domains_list'] = nodes_golden_power['domains'].apply(parse_domains)
nodes_golden_power['is_multidomain'] = nodes_golden_power['domains_list'].apply(lambda x: len(x) > 1)
nodes_golden_power['num_domains'] = nodes_golden_power['domains_list'].apply(len)
nodes_golden_power['domains_display'] = nodes_golden_power['domains_list'].apply(lambda x: ', '.join(x))

# Clean domain labels
def clean_label(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'^\d+\s+', '', text)
    text = text.replace('UNKNOWN', 'Altro').strip().upper()
    return text

# Enrich cluster names
def get_cluster_identity(df_nodes, cluster_id):
    cluster_data = df_nodes[df_nodes['global_cluster'] == cluster_id]
    
    # Extract dominant domain
    dom_list = [d.strip() for d in ";".join(cluster_data['domains']).split(";") if d.strip()]
    dominant_domain = pd.Series(dom_list).mode()[0] if dom_list else "Settore Trasversale"
    
    # Extract subdomain
    sub_list = [s.strip() for s in ";".join(cluster_data['subdomains']).split(";") if s.strip()]
    dominant_label = pd.Series(sub_list).mode()[0] if sub_list else f"Cluster {int(cluster_id)}"
    
    return dominant_domain, dominant_label

# Apply cluster naming
id_to_names = {}
for clus_id in nodes_golden_power['global_cluster'].unique():
    domain, label = get_cluster_identity(nodes_golden_power, clus_id)
    id_to_names[clus_id] = {'macro': domain, 'sub': label}

nodes_golden_power['macro_area'] = nodes_golden_power['global_cluster'].map(lambda x: id_to_names[x]['macro'])
nodes_golden_power['sub_community_name'] = nodes_golden_power['global_cluster'].map(lambda x: id_to_names[x]['sub'])
nodes_golden_power['macro_area_clean'] = nodes_golden_power['macro_area'].apply(clean_label)
nodes_golden_power['sub_name_clean'] = nodes_golden_power['sub_community_name'].apply(clean_label)

# Mark seed node
nodes_golden_power['is_seed'] = nodes_golden_power['celex'] == seed_celex

print(f"Data enriched:")
print(f"  • Multi-domain laws: {nodes_golden_power['is_multidomain'].sum()}/{len(nodes_golden_power)} ({100*nodes_golden_power['is_multidomain'].mean():.1f}%)")
print(f"  • Communities: {nodes_golden_power['global_cluster'].nunique()}")

### 1.6 Cross-Community Domain Analysis

**Problem**: Many domains span multiple functional communities, showing that administrative labels don't match functional reality.

In [ ]:
# Create Domain-Community Matrix
def create_domain_community_matrix(nodes_df):
    community_domain_data = []
    all_domains = sorted(set([d for domains in nodes_df['domains_list'] for d in domains]))
    
    for cluster_id in sorted(nodes_df['global_cluster'].unique()):
        cluster_data = nodes_df[nodes_df['global_cluster'] == cluster_id]
        cluster_label = cluster_data['macro_area_clean'].iloc[0]
        
        row = {
            'Community': cluster_label,
            'Total Laws': len(cluster_data)
        }
        
        for domain in all_domains:
            count = sum([domain in laws for laws in cluster_data['domains_list']])
            row[domain] = count
        
        community_domain_data.append(row)
    
    matrix_df = pd.DataFrame(community_domain_data)
    
    # Sort columns by frequency
    domain_cols = [col for col in matrix_df.columns if col not in ['Community', 'Total Laws']]
    domain_totals = matrix_df[domain_cols].sum().sort_values(ascending=False)
    sorted_domains = domain_totals.index.tolist()
    
    final_cols = ['Community', 'Total Laws'] + sorted_domains
    return matrix_df[final_cols]

domain_matrix = create_domain_community_matrix(nodes_golden_power)

print("Domain-Community Distribution Matrix:")
print("=" * 80)
print(domain_matrix.to_string(index=False))

# Identify cross-cutting domains
domain_to_communities = {}
for cluster_id in nodes_golden_power['global_cluster'].unique():
    cluster_data = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]
    cluster_label = cluster_data['macro_area_clean'].iloc[0]
    
    for domains in cluster_data['domains_list']:
        for domain in domains:
            if domain not in domain_to_communities:
                domain_to_communities[domain] = set()
            domain_to_communities[domain].add((cluster_id, cluster_label))

print("\n📊 Cross-Cutting Domains (spanning multiple communities):")
print("=" * 80)
cross_cutting = [(d, comms) for d, comms in domain_to_communities.items() if len(comms) > 1]
cross_cutting.sort(key=lambda x: len(x[1]), reverse=True)

for domain, communities in cross_cutting[:5]:
    print(f"\n{domain}: Spans {len(communities)} communities")
    for _, label in sorted(communities):
        n_laws = len(nodes_golden_power[
            (nodes_golden_power['macro_area_clean'] == label) &
            (nodes_golden_power['domains_list'].apply(lambda x: domain in x))
        ])
        print(f"  • {label} ({n_laws} laws)")

---
## Part 2: Visualization Solutions

Now we present two visualization approaches that address the problems identified above.

### 2.1 Community-Based Visualization

**Solution**: Group laws by citation-based functional communities, not administrative domains.

In [ ]:
# Calculate grid layout by community
pos = {}
unique_clusters = sorted(nodes_golden_power['global_cluster'].unique())
grid_cols = int(np.ceil(np.sqrt(len(unique_clusters))))
grid_spacing = 15

for i, cluster_id in enumerate(unique_clusters):
    row = i // grid_cols
    col = i % grid_cols
    center_x = col * grid_spacing
    center_y = -row * grid_spacing
    
    cluster_nodes = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]['celex'].tolist()
    if len(cluster_nodes) == 0:
        continue
    
    temp_G = nx.Graph()
    temp_G.add_nodes_from(cluster_nodes)
    radius = max(3, np.sqrt(len(cluster_nodes)) * 0.8)
    local_pos = nx.circular_layout(temp_G, center=(center_x, center_y), scale=radius)
    pos.update(local_pos)

# Add positions to dataframe
nodes_golden_power['x'] = nodes_golden_power['celex'].map(lambda c: pos[c][0] if c in pos else 0)
nodes_golden_power['y'] = nodes_golden_power['celex'].map(lambda c: pos[c][1] if c in pos else 0)

# Generate community labels
community_labels = {}
for cluster_id in unique_clusters:
    cluster_data = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]
    domain_counts = Counter()
    for domains in cluster_data['domains_list']:
        domain_counts.update(domains)
    
    top_domains = [d for d, _ in domain_counts.most_common(2) if d != 'UNKNOWN']
    
    if len(top_domains) >= 2:
        label = f"{top_domains[0]} & {top_domains[1]}"
    elif len(top_domains) == 1:
        label = top_domains[0]
    else:
        label = "Mixed Theme"
    
    n_multi = cluster_data['is_multidomain'].sum()
    if n_multi / len(cluster_data) > 0.5:
        label += " (Cross-cutting)"
    
    community_labels[cluster_id] = label

print("Functional Communities Detected:")
for cid, label in community_labels.items():
    n_laws = len(nodes_golden_power[nodes_golden_power['global_cluster'] == cid])
    print(f"  • {label}: {n_laws} laws")

In [ ]:
# Create community-based visualization
fig = go.Figure()

# Color palette
cluster_colors_palette = ['#2E86DE', '#EE5A6F', '#10AC84', '#F79F1F', '#A55EEA', 
                          '#54A0FF', '#FF6B6B', '#48DBFB', '#FF9FF3', '#1DD1A1']
cluster_colors = {c: cluster_colors_palette[i % len(cluster_colors_palette)] 
                 for i, c in enumerate(unique_clusters)}

# Add edges
edge_x, edge_y = [], []
for u, v in G_ego.edges():
    if u in pos and v in pos:
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='rgba(150,150,150,0.2)'),
    hoverinfo='none',
    showlegend=False
))

# Add community boundaries
shapes = []
annotations = []

for cluster_id in unique_clusters:
    subset = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]
    coords = [(pos[n][0], pos[n][1]) for n in subset['celex'] if n in pos]
    
    if not coords:
        continue
    
    cx = np.mean([c[0] for c in coords])
    cy = np.mean([c[1] for c in coords])
    radius = max(np.sqrt((np.array([c[0] for c in coords]) - cx)**2 + 
                        (np.array([c[1] for c in coords]) - cy)**2)) + 1.5
    
    shapes.append(dict(
        type="circle",
        xref="x", yref="y",
        x0=cx-radius, y0=cy-radius,
        x1=cx+radius, y1=cy+radius,
        fillcolor=cluster_colors[cluster_id],
        opacity=0.08,
        line=dict(color=cluster_colors[cluster_id], width=2.5, dash='dot'),
        layer='below'
    ))
    
    label = community_labels[cluster_id]
    n_laws = len(subset)
    n_multi = subset['is_multidomain'].sum()
    
    annotations.append(dict(
        x=cx,
        y=cy + radius + 3,
        text=f"<b>{label}</b><br><sub>{n_laws} laws ({n_multi} multi-domain)</sub>",
        showarrow=False,
        font=dict(size=14, color=cluster_colors[cluster_id], family="Arial Black"),
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor=cluster_colors[cluster_id],
        borderwidth=2,
        borderpad=6
    ))

# Add nodes by community
seed_node = nodes_golden_power[nodes_golden_power['celex'] == seed_celex]
other_nodes = nodes_golden_power[nodes_golden_power['celex'] != seed_celex]

for cluster_id in unique_clusters:
    subset = other_nodes[other_nodes['global_cluster'] == cluster_id]
    if len(subset) == 0:
        continue
    
    sizes = []
    border_widths = []
    border_colors = []
    
    for _, row in subset.iterrows():
        base_size = 12
        if row['is_multidomain']:
            sizes.append(base_size + row['num_domains'] * 2)
            border_widths.append(3)
            border_colors.append('#FFD700')
        else:
            sizes.append(base_size)
            border_widths.append(1.5)
            border_colors.append('white')
    
    hover_texts = []
    for _, row in subset.iterrows():
        hover_text = (
            f"<b>{row['title']}</b><br><br>"
            f"<b>CELEX:</b> {row['celex']}<br>"
            f"<b>Year:</b> {int(row['year'])}<br><br>"
            f"<b>Community:</b> {community_labels[cluster_id]}<br><br>"
            f"<b>Domains ({row['num_domains']}):</b><br>{row['domains_display']}"
        )
        hover_texts.append(hover_text)
    
    fig.add_trace(go.Scatter(
        x=subset['x'],
        y=subset['y'],
        mode='markers',
        name=f"{community_labels[cluster_id]} ({len(subset)})",
        marker=dict(
            size=sizes,
            color=cluster_colors[cluster_id],
            opacity=0.85,
            line=dict(width=border_widths, color=border_colors)
        ),
        text=hover_texts,
        hovertemplate='%{text}<extra></extra>',
        showlegend=True
    ))

# Add seed node
if not seed_node.empty and seed_celex in pos:
    seed_hover = (
        f"<b>{seed_node['title'].iloc[0]}</b><br><br>"
        f"<b>GOLDEN POWER - EU 2019/452</b><br>"
        f"<b>FDI Screening Framework</b><br><br>"
        f"<b>Domains ({seed_node['num_domains'].iloc[0]}):</b><br>"
        f"{seed_node['domains_display'].iloc[0]}"
    )
    
    fig.add_trace(go.Scatter(
        x=[seed_node['x'].iloc[0]],
        y=[seed_node['y'].iloc[0]],
        mode='markers+text',
        name='GOLDEN POWER (Seed)',
        marker=dict(
            size=35,
            color='#FF1744',
            symbol='star',
            line=dict(width=4, color='white')
        ),
        text=['★ GOLDEN POWER'],
        textposition='top center',
        textfont=dict(size=14, color='#FF1744', family='Arial Black'),
        hovertemplate=seed_hover + '<extra></extra>',
        showlegend=True
    ))

# Update layout
fig.update_layout(
    title=dict(
        text="<b>EU Law Network: Community-Based Visualization</b><br>" +
             "<sub>Grouped by Functional Communities (Citation-Based)</sub><br>" +
             "<sub style='color:#666;'>Gold borders = Multi-domain laws | Hover to see all domains</sub>",
        font=dict(size=20, family='Arial Black'),
        x=0.5,
        xanchor='center'
    ),
    shapes=shapes,
    annotations=annotations,
    showlegend=True,
    legend=dict(
        title=dict(text='<b>Functional Communities</b><br><sub>Click to hide/show</sub>'),
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.95)',
        x=1.02,
        y=0.5
    ),
    width=1600,
    height=1100,
    template="plotly_white",
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1),
    plot_bgcolor='#FAFAFA'
)

fig.show()

print("\n✅ Community-Based Visualization Complete")
print("This view shows laws grouped by WHY they're connected (citations)")
print("instead of WHAT domain they're administratively assigned to.")

### 2.2 Classification Comparison: EuroVoc vs LCGraph

**Solution**: Side-by-side comparison showing the difference between administrative (EuroVoc) and functional (LCGraph) classification.

In [ ]:
# Prepare data for comparison
nodes_golden_power['primary_domain'] = nodes_golden_power['domains_list'].apply(lambda x: x[0])
nodes_golden_power['is_multi'] = nodes_golden_power['domains_list'].apply(lambda x: len(x) > 1)

# Create color maps
all_domains_unique = sorted(set([d for domains in nodes_golden_power['domains_list'] for d in domains]))
color_palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
domain_colors = {d: color_palette[i % len(color_palette)] for i, d in enumerate(all_domains_unique)}

cluster_colors_list = ['#4285F4', '#EA4335', '#34A853', '#FBBC04', '#9C27B0']
cluster_colors_comp = {c: cluster_colors_list[i % len(cluster_colors_list)] 
                       for i, c in enumerate(unique_clusters)}

# Calculate layouts
pos_left = nx.spring_layout(G_ego, k=2.0, seed=42, iterations=50)

# Grid layout for right panel
pos_right = {}
grid_spacing_comp = 10
for i, cluster_id in enumerate(unique_clusters):
    row = i // grid_cols
    col = i % grid_cols
    center_x = col * grid_spacing_comp
    center_y = -row * grid_spacing_comp
    
    cluster_nodes = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]['celex'].tolist()
    if len(cluster_nodes) == 0:
        continue
    
    temp_G = nx.Graph()
    temp_G.add_nodes_from(cluster_nodes)
    radius = max(2.5, np.sqrt(len(cluster_nodes)) * 0.7)
    local_pos = nx.circular_layout(temp_G, center=(center_x, center_y), scale=radius)
    pos_right.update(local_pos)

In [ ]:
# Create comparison visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        '<b>EuroVoc:</b> A Priori Classification',
        '<b>LCGraph:</b> Citation-Based Clustering'
    ),
    horizontal_spacing=0.12,
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}]]
)

# LEFT PANEL: EuroVoc classification
# Edges
edge_x_left, edge_y_left = [], []
for u, v in G_ego.edges():
    if u in pos_left and v in pos_left:
        x0, y0 = pos_left[u]
        x1, y1 = pos_left[v]
        edge_x_left.extend([x0, x1, None])
        edge_y_left.extend([y0, y1, None])

fig.add_trace(
    go.Scatter(
        x=edge_x_left, y=edge_y_left,
        mode='lines',
        line=dict(width=0.5, color='rgba(150,150,150,0.25)'),
        hoverinfo='none',
        showlegend=False
    ),
    row=1, col=1
)

# Nodes by domain color
color_to_nodes = {}
for _, row in nodes_golden_power.iterrows():
    color = domain_colors[row['primary_domain']]
    if color not in color_to_nodes:
        color_to_nodes[color] = []
    color_to_nodes[color].append(row)

for color, node_list in color_to_nodes.items():
    node_subset = pd.DataFrame(node_list)
    x = [pos_left[n][0] for n in node_subset['celex'] if n in pos_left]
    y = [pos_left[n][1] for n in node_subset['celex'] if n in pos_left]
    
    sizes = [24 if celex == seed_celex else 11 for celex in node_subset['celex'] if celex in pos_left]
    borders = ['#FFD700' if is_multi else 'white' 
              for is_multi, celex in zip(node_subset['is_multi'], node_subset['celex']) 
              if celex in pos_left]
    border_widths = [2.5 if is_multi else 1.5 
                   for is_multi, celex in zip(node_subset['is_multi'], node_subset['celex']) 
                   if celex in pos_left]
    
    hover_data = []
    for _, row in node_subset.iterrows():
        if row['celex'] not in pos_left:
            continue
        domains_str = ', '.join(row['domains_list'])
        hover_data.append(
            f"Primary: {row['primary_domain']}<br>"
            f"All ({len(row['domains_list'])}): {domains_str}"
        )
    
    fig.add_trace(
        go.Scatter(
            x=x, y=y,
            mode='markers',
            marker=dict(
                size=sizes,
                color=color,
                line=dict(width=border_widths, color=borders),
                opacity=0.85
            ),
            text=node_subset['title'],
            customdata=hover_data,
            hovertemplate='<b>%{text}</b><br>%{customdata}<extra></extra>',
            showlegend=False
        ),
        row=1, col=1
    )

# RIGHT PANEL: LCGraph clustering
# Edges
edge_x_right, edge_y_right = [], []
for u, v in G_ego.edges():
    if u in pos_right and v in pos_right:
        x0, y0 = pos_right[u]
        x1, y1 = pos_right[v]
        edge_x_right.extend([x0, x1, None])
        edge_y_right.extend([y0, y1, None])

fig.add_trace(
    go.Scatter(
        x=edge_x_right, y=edge_y_right,
        mode='lines',
        line=dict(width=0.6, color='rgba(100,100,100,0.3)'),
        hoverinfo='none',
        showlegend=False
    ),
    row=1, col=2
)

# Cluster boundaries
shapes = []
for cluster_id in unique_clusters:
    subset = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]
    coords = [(pos_right[n][0], pos_right[n][1]) for n in subset['celex'] if n in pos_right]
    if not coords:
        continue
    
    cx = np.mean([c[0] for c in coords])
    cy = np.mean([c[1] for c in coords])
    radius = max(np.sqrt((np.array([c[0] for c in coords]) - cx)**2 + 
                        (np.array([c[1] for c in coords]) - cy)**2)) + 0.8
    
    shapes.append(dict(
        type="circle",
        xref="x2", yref="y2",
        x0=cx-radius, y0=cy-radius,
        x1=cx+radius, y1=cy+radius,
        fillcolor=cluster_colors_comp[cluster_id],
        opacity=0.1,
        line=dict(color=cluster_colors_comp[cluster_id], width=2.5, dash='dot'),
        layer='below'
    ))

# Nodes by cluster
for cluster_id in unique_clusters:
    subset = nodes_golden_power[nodes_golden_power['global_cluster'] == cluster_id]
    cluster_name = subset['macro_area_clean'].mode()[0]
    
    x = [pos_right[n][0] for n in subset['celex'] if n in pos_right]
    y = [pos_right[n][1] for n in subset['celex'] if n in pos_right]
    sizes = [26 if celex == seed_celex else 13 for celex in subset['celex'] if celex in pos_right]
    
    fig.add_trace(
        go.Scatter(
            x=x, y=y,
            mode='markers',
            name=cluster_name,
            marker=dict(
                size=sizes,
                color=cluster_colors_comp[cluster_id],
                opacity=0.9,
                line=dict(width=2, color='white')
            ),
            text=subset['title'],
            customdata=subset['domains_list'].apply(lambda x: ', '.join(x)),
            hovertemplate='<b>%{text}</b><br>Themes: %{customdata}<extra></extra>',
            showlegend=True
        ),
        row=1, col=2
    )

# Update axes
fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

# Update layout
fig.update_layout(
    title=dict(
        text='<b>Classification Comparison: EuroVoc vs LCGraph</b><br>' +
             '<sub>Golden Power Regulation (2019/452)</sub>',
        x=0.5,
        y=0.98,
        xanchor='center',
        yanchor='top',
        font=dict(size=18, family='Arial')
    ),
    shapes=shapes,
    height=750,
    width=1700,
    showlegend=True,
    legend=dict(
        title='<b>Functional Clusters</b>',
        x=1.01,
        y=0.5,
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='gray',
        borderwidth=1
    ),
    plot_bgcolor='#FAFAFA',
    hovermode='closest',
    margin=dict(t=140, b=80, l=40, r=180)
)

# Add annotations
fig.add_annotation(
    text=f"<b>LEFT:</b> {len(all_domains_unique)} EuroVoc domains<br><i>Gold border = multi-domain</i>",
    xref="paper", yref="paper",
    x=0.20, y=-0.06,
    showarrow=False,
    font=dict(size=12, color='#d32f2f', family='Arial'),
    bgcolor='rgba(255,255,255,0.95)',
    bordercolor='#d32f2f',
    borderwidth=2,
    borderpad=8
)

fig.add_annotation(
    text=f"<b>RIGHT:</b> {len(unique_clusters)} functional clusters<br><i>Based on citations</i>",
    xref="paper", yref="paper",
    x=0.80, y=-0.06,
    showarrow=False,
    font=dict(size=12, color='#388e3c', family='Arial'),
    bgcolor='rgba(255,255,255,0.95)',
    bordercolor='#388e3c',
    borderwidth=2,
    borderpad=8
)

fig.show()

print("\n✅ Classification Comparison Complete")
print("LEFT: Laws scattered by administrative domains (fragmented view)")
print("RIGHT: Laws grouped by functional relationships (unified view)")

---
## Summary and Export

### Key Findings

1. **Domain Fragmentation**: Golden Power touches 15+ domains despite being a unified regulatory framework
2. **High Overlap**: 60%+ of laws have multiple domain assignments
3. **Cross-Cutting Themes**: Major domains (Trade, Finance, EU) span multiple functional communities
4. **Citation-Based Coherence**: Network analysis reveals 5 functional communities that better reflect legislative architecture

### Conclusion

Traditional classification systems (EuroVoc) obscure functional relationships by imposing a priori administrative categories. Citation-based network analysis (LCGraph) reveals the true structure of legislative relationships.

In [ ]:
# Save enriched data for future use
output_path = os.path.join('..', 'data', 'processed', 'nodes_enriched.csv')
nodes_golden_power.to_csv(output_path, index=False)

print(f"✅ Enriched data saved to: {output_path}")
print(f"   • {len(nodes_golden_power)} nodes")
print(f"   • {nodes_golden_power['global_cluster'].nunique()} communities")
print(f"   • {nodes_golden_power['is_multidomain'].sum()} multi-domain laws")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)